# 🎨 ComfyUI на GPU для бота «Лея» (Google Colab, бесплатно)

Этот ноутбук запускает **ComfyUI на бесплатном GPU (Tesla T4)** и открывает доступ
для вашего бота. Пока работает эта сессия — бот рисует картинки БЫСТРО (5–30 секунд).

## Как пользоваться:
1. Нажмите **Файл → Сохранить копию на Диске** (или просто запускайте).
2. Меню **Среда выполнения → Сменить среду выполнения** → **T4 GPU** → Сохранить.
3. Запускайте ячейки по очереди (Shift+Enter) или **Среда выполнения → Выполнить всё**.
4. В конце появится **адрес** (вида `https://xxxx.localtunnel.me` или `https://xxxx.loca.lt`).
5. Этот адрес впишите в файл `.env` бота: `COMFYUI_BASE_URL=https://xxxx.localtunnel.me`
   (и перезапустите бота).
6. Когда закончите — остановите сессию (меню → Среда выполнения → Прервать), чтобы не тратить лимит.

> ⚠️ Сессия Colab живёт несколько часов и прерывается при бездействии.
> Нужны картинки → запустили ноутбук → получили адрес → рисуете → остановили.

In [ ]:
# 1. Проверяем GPU
import subprocess, sys
!nvidia-smi
print("GPU OK")

In [ ]:
# 2. Скачиваем ComfyUI (если ещё нет)
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
print("ComfyUI готов")

In [ ]:
# 3. Ставим зависимости (первый раз ~3-5 минут)
!pip install -q -r /content/ComfyUI/requirements.txt
print("Зависимости готовы")

In [ ]:
# 4. Скачиваем модель-«художника» (majicMIX realistic, NSFW 18+, ~2 ГБ)
#    Если ссылка не работает — замените на другую модель из списка ниже.
import os
ckpt_dir = '/content/ComfyUI/models/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
target = f'{ckpt_dir}/majicmixRealistic_v7.safetensors'
if not os.path.exists(target):
    !curl -L -o "{target}" "https://huggingface.co/lllyasviel/fav_models/resolve/main/fav/majicmixRealistic_v7.safetensors"
    # Запасной вариант (если первый не скачался):
    # !curl -L -o "{target}" "https://huggingface.co/digiplay/majicMIX_realistic_v7/resolve/main/majicmixRealistic_v7.safetensors"
print("Модель готова:", os.path.getsize(target)//(1024**3), "ГБ")

In [ ]:
# 5. Запускаем ComfyUI на GPU (в фоне)
import subprocess, time, os
log = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=log, stderr=log)
print("ComfyUI запускается...")
for i in range(60):
    time.sleep(2)
    try:
        import urllib.request
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print("✅ ComfyUI работает!")
        break
    except Exception:
        pass
else:
    print("ComfyUI не поднялся за 2 минуты — смотрите лог:")
    print(open('/content/comfyui.log').read()[-3000:])

In [ ]:
# 6. Открываем доступ для бота через туннель
#    localtunnel (проще) или loca.lt (запасной)
import subprocess, threading, time

def run_tunnel():
    subprocess.run(['npx', '-y', 'localtunnel', '--port', '8188'])

try:
    # пробуем localtunnel
    t = threading.Thread(target=run_tunnel, daemon=True)
    t.start()
    time.sleep(12)
    print("Туннель запущен. Ищите адрес вида:")
    print("  https://xxxx.loca.lt  —  это и есть COMFYUI_BASE_URL")
    print("Если адрес не появился — используйте запасную ячейку ниже.")
except Exception as e:
    print("localtunnel не сработал:", e)
    print("Используйте запасную ячейку ниже.")

In [ ]:
# 7. ЗАПАСНОЙ туннель: cloudflared (если localtunnel не дал адрес)
!which cloudflared || (curl -L -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared)
!cloudflared tunnel --url http://127.0.0.1:8188


## 📋 Что делать с адресом

1. Скопируйте адрес из вывода (например `https://xxxx.loca.lt`).
2. На сервере (или ПК) откройте `.env` бота и поменяйте:
   ```
   COMFYUI_BASE_URL=https://xxxx.loca.lt
   ```
3. Перезапустите бота (`sudo systemctl restart project-lady` на Oracle).
4. Отправьте боту `/photo ...` — картинка придёт за 5–30 секунд!

## ⚠️ Важно
- Сессия Colab живёт несколько часов; при бездействии — прерывается.
- Когда закончили — остановите сессию, чтобы не расходовать лимит бесплатного GPU.
- Для постоянной работы картинок держите ноутбук запущенным.